## Correlation of evaluations

This notebook shows how to load conversations with their scores from CSV files and determine the correlation across these scores.
Here we use the build-in correlation function from ```pandas``` which supports **Pearson**, **Kendall** and **Spearman** methods.

We assume that the scores are kept in CSV files that represent the turns on each row with a turn identifier, the speaker and a predefined column with the score, 
e.g. with coherence scores:

```
Turn	Speaker	Response	llm_coherence
fa29f41f-7bea-42b1-bc64-17c6a1bd320e	LEOLANI	Yo Do you want to talk to me Luis?	0.2
4df4c74e-44cf-471a-b0cb-5233d96dbb21	SPEAKER	Yes	0.9
d9f4848a-87ab-4599-b6ef-e5bff1149e26	LEOLANI	I have nothing more to say.	0.3
```

or with human evaluations.

```
Turn	Speaker	Response	Reference Response	Overall Human Rating	Engaging	Specific	Relevant	Correct	Understandable	Coherent
fa29f41f-7bea-42b1-bc64-17c6a1bd320e	LEOLANI	Yo Do you want to talk to me Luis?		3	2	2	3	4	4	1
4df4c74e-44cf-471a-b0cb-5233d96dbb21	SPEAKER	Yes		5	2	3	4	5	5	5
d9f4848a-87ab-4599-b6ef-e5bff1149e26	LEOLANI	I have nothing more to say.		1	1	2	1	3	4	1
```

The functions below, will read series of CSV files, assuming that each has the same columns for the Turn, the Speaker and the Reponse.
You need to specify the name of the column that contains the scores to compare. It will compare the scores for each turn that has the same turn id.
You can define multiple columns per CSV file by rpeating the file with a different column name.
For example, when the human evaluation spans several metrics, you can select each metric columns separately.
In fact you can also test the correlation across the different human metrics.

The correlations are printed to the screen and saved to a heatmap in the evaluation folder of a scenario.

## Prerequisites

In [34]:
# !pip install pandas
# !pip install matplotlib
#!pip install seaborn

In [1]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

The next functions load the CSV files and keep the necessary columns for each. They create a joined dataframe for which the correlations is extracted.
Note that the CSV files should:

* have the same conversations represented by a sequence of rows with their Turn id, the Speaker and Response
* have a unique name for their column that is different from the any other column against which to compare

You may rename the column names with the evaluation score in the CSV files and adapt the code accordingly.

In [2]:
def read_evaluations(file, column):
    columns = ["Turn", "Speaker", "Response", column]
    evaluations = []
    try:
        df = pd.read_csv(file, header=0, undex_col='Turn')
    except:
        try:
            df = pd.read_csv(file, header=0, index_col='Turn', sep=';')
        except:
            print(f"Could not load {file}")
            df = pd.DataFrame()
            # continue
    columns_to_keep = [c for c in columns if c in df.columns]
    df = df[columns_to_keep]
    evaluations.append(df)
    return df

def correlate(files, cols):
    evaluations = []
    for file, col in zip(files,cols):
        file_evaluations = read_evaluations(file, col)
        evaluations.append(file_evaluations)
    full_df = pd.concat(evaluations, axis=1)
    # Compute correlations
    corr_df = full_df.corr(method='pearson', numeric_only=True)
    return corr_df

def plot_correlations(df_to_plot, mask, scenario, evaluation_folder):
    # Plot
    plt.figure()
    plt.xticks(fontsize=3)
    plt.yticks(fontsize=3)

    g = sns.heatmap(df_to_plot, mask=mask, annot=False, fmt=".2f",
                    cmap="YlGnBu", cbar_kws={"shrink": .3, "location": "top"},
                    cbar=True, center=0,
                    square=True)

    # Save
    plot_file = os.path.join(evaluation_folder, scenario+"_correlations_heatmap.png")
    g.figure.savefig(plot_file, dpi=300, transparent=False, bbox_inches='tight')
    plt.close()
    print(f"\tSaved to file: {plot_file}")

Next we demonstrate how we can select multiple columns from a CSV file with manual human evaluations and a coherence column from an LLM-as-judge evaluation.

In [4]:
EMISSOR="../emissor"
SCENARIO="b387db06-934e-405b-9d4e-f7e5c27b440a"

file1= SCENARIO+"_manual_evaluation_scored.csv"
col1a= "Overall Human Rating"
col1b= "Specific"
col1c= "Coherence"
col1d= "Understandable"
col1e= "Relevant"
col1f= "Engaging"

file2= SCENARIO+"_llm_judge_evaluation.csv"
col2= "llm_coherence"

file3= SCENARIO+"_likelihood_evaluation.csv"
col3= "llh"

evaluation_folder = os.path.join(EMISSOR, SCENARIO, 'evaluation')
if not os.path.exists(evaluation_folder):
    print("Cannot find the evaluation folder", evaluation_folder)
else:
    file1_path = os.path.join(evaluation_folder, file1)
    file2_path = os.path.join(evaluation_folder, file2)
    file3_path = os.path.join(evaluation_folder, file3)
    files = [file1_path, file1_path, file1_path, file1_path, file1_path, file1_path, file2_path, file3_path]
    cols = [col1a, col1b, col1c, col1d, col1e, col1f, col2, col3]
    corr_df = correlate(files, cols)
    print(corr_df)
    plot_correlations(corr_df, None, SCENARIO, evaluation_folder)
    csv_file = os.path.join(evaluation_folder, SCENARIO+"_correlations.csv")


                      Overall Human Rating  Specific  Understandable  \
Overall Human Rating              1.000000  0.285750        0.721688   
Specific                          0.285750  1.000000        0.329956   
Understandable                    0.721688  0.329956        1.000000   
Relevant                          0.875000  0.190500        0.721688   
Engaging                          0.501115  0.526104        0.231455   
llm_coherence                     0.448013  0.459397        0.517321   
llh                              -0.285117  0.031753        0.192472   

                      Relevant  Engaging  llm_coherence       llh  
Overall Human Rating  0.875000  0.501115       0.448013 -0.285117  
Specific              0.190500  0.526104       0.459397  0.031753  
Understandable        0.721688  0.231455       0.517321  0.192472  
Relevant              1.000000  0.400892       0.248896 -0.153485  
Engaging              0.400892  1.000000      -0.097563 -0.441109  
llm_coherence  

In [3]:
EMISSOR="../emissor"
SCENARIO = "2ad0c8b9-4e62-4f11-809c-57bf76487f95"

file1= SCENARIO+"_graph_evaluation.csv"
col1a= "GROUP A - Average degree"
col1b= "GROUP A - Centrality entropy"
col1c= "GROUP A - Sparseness"
col1d= "ROUP C - Total triples"
col1e= "GROUP C - Total world instances"
col1f= "GROUP C - Ratio claim to triples"

file2= SCENARIO+"_llm_judge_evaluation.csv"
col2= "llm_coherence"

file3= SCENARIO+"_likelihood_evaluation.csv"
col3= "llh"

evaluation_folder = os.path.join(EMISSOR, SCENARIO, 'evaluation')
if not os.path.exists(evaluation_folder):
    print("Cannot find the evaluation folder", evaluation_folder)
else:
    file1_path = os.path.join(evaluation_folder, file1)
    file2_path = os.path.join(evaluation_folder, file2)
    file3_path = os.path.join(evaluation_folder, file3)
    files = [file1_path, file1_path, file1_path, file1_path, file1_path, file1_path, file2_path, file3_path]
    cols = [col1a, col1b, col1c, col1d, col1e, col1f, col2, col3]
    corr_df = correlate(files, cols)
    print(corr_df)
    plot_correlations(corr_df, None, SCENARIO, evaluation_folder)
    csv_file = os.path.join(evaluation_folder, SCENARIO+"_correlations.csv")

                                  GROUP A - Average degree  \
GROUP A - Average degree                          1.000000   
GROUP A - Centrality entropy                      0.879049   
GROUP A - Sparseness                             -0.752248   
GROUP C - Total world instances                   0.717870   
GROUP C - Ratio claim to triples                 -0.059682   
llm_coherence                                     0.034478   
llh                                               0.100547   

                                  GROUP A - Centrality entropy  \
GROUP A - Average degree                              0.879049   
GROUP A - Centrality entropy                          1.000000   
GROUP A - Sparseness                                  0.720071   
GROUP C - Total world instances                       0.640542   
GROUP C - Ratio claim to triples                     -0.322449   
llm_coherence                                         0.063545   
llh                                      

## End of notebook